# Stage D / NB 20 — interpretability and failure analysis (E10)

Protocol reference: family **E10**; referee point **R1.4** ("show correctly detected low,
moderate, severe, and failure cases").

## The sampling rule is fixed before anyone looks

Protocol E10 pre-registers it, and this notebook executes it verbatim. Seed 42.

- 4 cases per severity band (0 / 1–10 / 11–18 / 19–24) × {correct, incorrect}
- the 10 largest absolute mRALE errors
- **all** cases with a localization fallback
- **all** agent-conflict cases

Choosing panels after seeing results is how a figure becomes an advertisement. The rule is
executed by code and the selection is stamped with a hash covering the rule, locked case-table
fingerprint, and selected image keys, so a reader can verify that the panels shown are the
panels the rule selects from the exact audited predictions.

## PCR-positive with mRALE 0 is a label/opacity discordance

The most important category in the taxonomy. This cohort's label is PCR, and a PCR-positive
patient can have an mRALE opacity score of 0. A model mRALE estimate of 0 is then correct for
the opacity endpoint, while its COVID decision remains separately judged against PCR.

The taxonomy separates it out, and the notebook reports what fraction of apparent detection
failures fall into it. The result quantifies label/opacity discordance; it does not assert that
every other radiographic finding is absent or prove a universal image-based detection ceiling.

## Dual coding is a protocol requirement, not a nicety

E10b asks for two independent coders and Cohen's κ. Code cannot supply human judgement, so this
notebook produces the **coding sheet** with an automatic first pass, and the gate refuses to
report a taxonomy as dual-coded until two human codings exist. A single automatic coding
presented as a taxonomy would misrepresent the method.

## Outputs (under `stage_D/nb20_interpretability/`)
`case_panels/*.png`, `case_selection.csv`, `failure_taxonomy.csv`, `coding_sheet.csv`,
`dual_coding_kappa.json`, `grounding_metrics.csv`, `reader_study_protocol.md`, `gate_nb20.json`.

## 1. Imports and locked artifacts

In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# Metric definitions are shared with Stage B/C. Every number in the manuscript must come from
# the same code that produced the arm tables, or the tables and the statistics disagree.
_SEARCH = [Path.cwd(), Path.cwd().parent, Path.cwd().parent / "stage_B",
           Path.cwd().parent.parent / "notebooks" / "stage_B"]
for _candidate in _SEARCH:
    if (_candidate / "cxr_metrics.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError(f"cxr_metrics.py not found. Searched: {_SEARCH}")
import cxr_metrics as cm

# Stage D's own statistics module: bootstrap indices drawn once, DeLong, McNemar, Holm, TOST,
# and the rule that a p-value cannot exist without its metadata.
for _candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent / "stage_D",
                   Path.cwd().parent.parent / "notebooks" / "stage_D"]:
    if (_candidate / "stage_d_stats.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("stage_d_stats.py not found; it must sit beside these notebooks.")
import stage_d_stats as sd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FALLBACK_STAGE_A = Path("/data/liangz2/openi/midrc/tetci_resubmit/stage_A")
for _candidate in [FALLBACK_STAGE_A / "nb00_environment" / "stage_a_paths.json",
                   Path.cwd() / "stage_a_paths.json",
                   Path.cwd().parent / "stage_A" / "nb00_environment" / "stage_a_paths.json"]:
    if _candidate.is_file():
        stage_paths = json.loads(_candidate.read_text(encoding="utf-8"))
        print("Path contract:", _candidate)
        break
else:
    raise FileNotFoundError("stage_a_paths.json not found. Run Stage A NB 00 first.")

PROJECT_ROOT = Path(stage_paths["project_root"])
STAGE_ROOT = Path(stage_paths["stage_root"])
STAGE_A_DIR = Path(stage_paths["stage_a_dir"])
STAGE_B_DIR = STAGE_ROOT / "stage_B"
STAGE_C_DIR = STAGE_ROOT / "stage_C"
STAGE_D_DIR = STAGE_ROOT / "stage_D"
STAGE_D_DIR.mkdir(parents=True, exist_ok=True)
NB01_DIR = Path(stage_paths["nb_output_dirs"]["nb01_inventory"])
NB02_DIR = Path(stage_paths["nb_output_dirs"]["nb02_folds"])
NB03_DIR = Path(stage_paths["nb_output_dirs"]["nb03_external"])
NB04_DIR = Path(stage_paths["nb_output_dirs"]["nb04_localization"])
FOLD_DEF_DIR = NB02_DIR / "fold_definitions"
MODEL_REVISIONS = stage_paths.get("model_revisions", {})

N_FOLDS = 5
N_BOOTSTRAP = sd.BOOTSTRAP_REPLICATES
MAX_SESSION_HOURS = 35.0      # Biowulf limit is 36 h; guard section boundaries
SESSION_DEADLINE = sd.make_session_deadline(MAX_SESSION_HOURS)

print("Stage D output:", STAGE_D_DIR)
print(f"Bootstrap: {N_BOOTSTRAP} patient-level replicates, seed {sd.BOOTSTRAP_SEED}")
print(f"Soft stop: {MAX_SESSION_HOURS:.1f} h after setup; checks occur between sections")

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 20 before code cell 3")

def load_folds():
    path = FOLD_DEF_DIR / "midrc_folds_v2.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"{path} not found. Run Stage A NB 02 first. Do NOT fall back to the legacy "
            "multi_task_CV folds: they leak at study level.")
    frame = pd.read_csv(path)
    frame["image_key"] = "MIDRC::" + frame["filename"].astype(str)
    return frame


# Every internal out-of-fold prediction file Stage B and Stage C can produce.
# (label, family, directory, filename). `family` drives the multiplicity families of 8.6.
INTERNAL_SOURCES = [
    ("A5_cxformer",        "E0",  STAGE_B_DIR / "nb05_frozen_encoder",     "predictions_frozen.jsonl"),
    ("E0g_conventional",   "E0",  STAGE_B_DIR / "nb06_conventional",       "predictions_conventional.jsonl"),
    ("E0_zeroshot",        "E0",  STAGE_B_DIR / "nb07_zeroshot",           "predictions_zeroshot.jsonl"),
    ("A6_biomedclip",      "E0",  STAGE_B_DIR / "nb08_biomedclip_entity",  "predictions_entity_probe.jsonl"),
    ("A2_medgemma_lora",   "E0",  STAGE_B_DIR / "nb09_medgemma_lora",      "predictions_medgemma_lora.jsonl"),
    ("A3_qwen_lora",       "E0",  STAGE_B_DIR / "nb10_qwen_lora",          "predictions_qwen_lora.jsonl"),
    ("A4_nvreason",        "E0",  STAGE_B_DIR / "nb11_nvreason",           "predictions_nvreason.jsonl"),
    ("E4_anatomy",         "E4",  STAGE_B_DIR / "nb12_anatomy_aware",      "anatomy_aware_predictions.jsonl"),
    ("E7_fusion",          "E7",  STAGE_C_DIR / "nb14_fusion",             "fusion_predictions.jsonl"),
]
EXTERNAL_SOURCES = [
    ("A5_cxformer",      STAGE_B_DIR / "nb05_frozen_encoder",    "external_predictions.jsonl"),
    ("E0g_conventional", STAGE_B_DIR / "nb06_conventional",      "external_predictions.jsonl"),
    ("A6_biomedclip",    STAGE_B_DIR / "nb08_biomedclip_entity", "external_predictions.jsonl"),
    ("A2_medgemma_lora", STAGE_B_DIR / "nb09_medgemma_lora",     "external_predictions.jsonl"),
    ("A3_qwen_lora",     STAGE_B_DIR / "nb10_qwen_lora",         "external_predictions.jsonl"),
]


def discover_arms(log=print):
    """
    Build the arm table from whatever Stage B and Stage C actually produced.

    Absence is recorded, never inferred: a notebook that has not run is a different thing from
    an arm that produced nothing, and the two have different remedies.
    """
    rows, availability = [], []
    for label, family, directory, filename in INTERNAL_SOURCES:
        path = directory / filename
        if not path.is_file():
            availability.append({"source": label, "family": family, "path": str(path),
                                 "status": "MISSING", "n_rows": 0, "n_arms": 0,
                                 "reason": "prediction file not found; notebook not yet run"})
            continue
        found = cm.read_jsonl(path)
        arms = sorted({str(r.get("arm", label)) for r in found})
        availability.append({"source": label, "family": family, "path": str(path),
                             "status": "OK", "n_rows": len(found), "n_arms": len(arms),
                             "reason": ""})
        for row in found:
            row["_source"] = label
            row["_family"] = family
            row["arm"] = str(row.get("arm", label))
            rows.append(row)

    # Stage C reasoner arms live in one file per roster.
    reasoner_dir = STAGE_C_DIR / "nb15_reasoner"
    reasoner_files = sorted(reasoner_dir.glob("reasoner_predictions_*.jsonl"))
    if reasoner_files:
        for path in reasoner_files:
            arm = path.stem.replace("reasoner_predictions_", "")
            found = cm.read_jsonl(path)
            family = "E1" if arm.startswith("E1") else "E7"
            availability.append({"source": f"reasoner/{arm}", "family": family,
                                 "path": str(path), "status": "OK", "n_rows": len(found),
                                 "n_arms": 1, "reason": ""})
            for row in found:
                row["_source"] = "reasoner"
                row["_family"] = family
                row["arm"] = arm
                rows.append(row)
    else:
        availability.append({"source": "reasoner", "family": "E1",
                             "path": str(reasoner_dir / "reasoner_predictions_*.jsonl"),
                             "status": "MISSING", "n_rows": 0, "n_arms": 0,
                             "reason": "NB 15 has not produced any roster predictions"})
    for entry in availability:
        log(f"  [{entry['status']:<7}] {entry['source']:<24} rows={entry['n_rows']:<7} "
            f"arms={entry['n_arms']}")
    return rows, pd.DataFrame(availability)

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 20 before code cell 4")

NB20_DIR = STAGE_D_DIR / "nb20_interpretability"
PANEL_DIR = NB20_DIR / "case_panels"
for directory in (NB20_DIR, PANEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)
NB17_DIR = STAGE_D_DIR / "nb17_statistics"        # READ ONLY
NB13_DIR = STAGE_C_DIR / "nb13_registry"          # READ ONLY
NB15_DIR = STAGE_C_DIR / "nb15_reasoner"          # READ ONLY


def require_upstream_gate(directory, notebook_number):
    candidates = [directory / f"gate_nb{notebook_number:02d}.json",
                  directory / f"gate_nb{notebook_number}.json"]
    path = next((candidate for candidate in candidates if candidate.is_file()), candidates[0])
    if not path.is_file():
        raise FileNotFoundError(
            f"Required upstream gate is missing: {path}. Run NB {notebook_number} to "
            "completion before NB 20; partial outputs cannot support qualitative audit.")
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not bool(payload.get("passed", False)):
        raise RuntimeError(
            f"NB {notebook_number} did not pass its gate: {payload.get('failures', [])}")
    return {"path": str(path), "passed": True}


UPSTREAM_GATES = {
    "NB04": require_upstream_gate(NB04_DIR, 4),
    "NB13": require_upstream_gate(NB13_DIR, 13),
    "NB15": require_upstream_gate(NB15_DIR, 15),
    "NB17": require_upstream_gate(NB17_DIR, 17),
}

nb17_config_path = NB17_DIR / "run_config.json"
if not nb17_config_path.is_file():
    raise FileNotFoundError(
        f"{nb17_config_path} not found. NB 17 defines the reference arm whose failures this "
        "notebook analyses; run it first. If the directory does not exist at all, check "
        "stage_a_paths.json rather than editing this path.")
nb17_config = json.loads(nb17_config_path.read_text(encoding="utf-8"))
nb15_config_path = NB15_DIR / "run_config.json"
metrics_path = NB17_DIR / "all_metrics_with_ci.csv"
for required in [nb15_config_path, metrics_path]:
    if not required.is_file():
        raise FileNotFoundError(f"Required locked artifact is missing: {required}")
nb15_config = json.loads(nb15_config_path.read_text(encoding="utf-8"))
REFERENCE_ARM = str(nb17_config["reference_arm"])
if str(nb15_config.get("full_roster_arm") or "") != REFERENCE_ARM:
    raise RuntimeError(
        "NB 15 and NB 17 disagree on the locked full-roster reference arm. "
        "Re-run NB 17 after the completed NB 15.")
locked_metrics = pd.read_csv(metrics_path)
if int((locked_metrics["arm"].astype(str) == REFERENCE_ARM).sum()) != 1:
    raise RuntimeError(
        f"NB 17 must contain exactly one metric row for {REFERENCE_ARM!r}.")
print(f"Analysing failures of the reference arm: {REFERENCE_ARM}")

# ---- The pre-registered sampling rule (protocol E10), fixed before any result is seen -----
SAMPLING_RULE = OrderedDict([
    ("per_band_per_outcome", 4),      # 4 x 4 bands x {correct, incorrect}
    ("largest_errors", 10),
    ("all_localization_fallbacks", True),
    ("all_agent_conflicts", True),
    ("correct_within", 2),            # "correct" means within 2 mRALE points
    ("seed", 42),
])
RULE_HASH = hashlib.sha256(
    json.dumps(SAMPLING_RULE, sort_keys=True).encode("utf-8")).hexdigest()[:16]
print(f"Sampling rule hash: {RULE_HASH}")
print(json.dumps(SAMPLING_RULE, indent=2))

## 2. Assemble the case table

One row per image for the reference arm, joined to ground truth, the agent panel, the
localization record and the reasoner's stated justification. Everything a panel needs, and
everything the taxonomy is coded against.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 20 before code cell 6")

folds = load_folds()
if folds["image_key"].astype(str).duplicated().any():
    duplicate_keys = folds.loc[folds["image_key"].astype(str).duplicated(keep=False),
                               "image_key"].astype(str).unique()[:5].tolist()
    raise RuntimeError(f"NB 02 fold table contains duplicate image keys: {duplicate_keys}")
truth = folds.set_index(folds["image_key"].astype(str))
raw_rows, _ = discover_arms(log=lambda *a: None)

def strict_bool(value, default=False):
    if value is None or cm.is_missing(value):
        return default
    if isinstance(value, str):
        return value.strip().lower() in {"1", "true", "yes", "y"}
    return bool(value)


def covid_label(value):
    if value is None or cm.is_missing(value):
        return None
    text = str(value).strip().lower()
    if text in {"yes", "positive", "pos", "1", "true"}:
        return "Yes"
    if text in {"no", "negative", "neg", "0", "false"}:
        return "No"
    return None


def agent_names(value):
    if isinstance(value, dict):
        return sorted(str(key) for key in value)
    if isinstance(value, (list, tuple, set)):
        return sorted(str(item.get("agent", item) if isinstance(item, dict) else item)
                      for item in value)
    return []

reference_candidates = [r for r in raw_rows if r["arm"] == REFERENCE_ARM
                        and str(r.get("image_key", "")) in truth.index]
reference_counts = Counter(str(row["image_key"]) for row in reference_candidates)
duplicates = [key for key, count in reference_counts.items() if count != 1]
if duplicates:
    raise RuntimeError(f"Reference arm has duplicate prediction rows: {duplicates[:5]}")
reference_rows = {str(r["image_key"]): r for r in reference_candidates}
missing_reference = sorted(set(truth.index.astype(str)) - set(reference_rows))
if missing_reference:
    raise RuntimeError(f"Reference arm is missing {len(missing_reference)} locked images, "
                       f"e.g. {missing_reference[:5]}.")
print(f"Reference-arm predictions: {len(reference_rows):,}")

# Agent panels come only from NB 13's fold-locked selected arm for each image.
registry = None
for path in [NB13_DIR / "agent_registry.parquet", NB13_DIR / "agent_registry.csv"]:
    if path.is_file():
        registry = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_csv(path)
        break
if registry is None or "selected_for_outer_fold" not in registry.columns:
    raise FileNotFoundError("NB 13 selected agent registry is required for deterministic panels.")
selected_registry = registry[registry["selected_for_outer_fold"].map(strict_bool)].copy()
duplicate_agents = selected_registry.duplicated(["image_key", "agent"], keep=False)
if duplicate_agents.any():
    raise RuntimeError("NB 13 selected registry has more than one arm per image/agent.")
agent_totals = defaultdict(dict)
for row in selected_registry.to_dict("records"):
    if not cm.is_missing(row.get("mrale_total")):
        agent_totals[str(row["image_key"])][str(row["agent"])] = float(row["mrale_total"])

# Localization records: which cases fell back to a heuristic box.
views = pd.DataFrame()
if (NB04_DIR / "view_index.csv").is_file():
    views = pd.read_csv(NB04_DIR / "view_index.csv")
    if views["image_key"].astype(str).duplicated().any():
        raise RuntimeError("NB 04 view_index.csv contains duplicate image_key rows.")
    views = views.set_index(views["image_key"].astype(str))
    print(f"Localization index: {len(views):,} images")
else:
    raise FileNotFoundError(
        f"{NB04_DIR / 'view_index.csv'} is missing despite NB 04's passing gate.")
missing_views = sorted(set(reference_rows) - set(views.index.astype(str)))

# The reasoner's structured justification per case (NB 15's traces).
traces = {}
trace_path = NB15_DIR / "reasoning_traces.jsonl"
if trace_path.is_file():
    for row in cm.read_jsonl(trace_path):
        if str(row.get("arm")) != REFERENCE_ARM:
            continue
        key = str(row.get("image_key"))
        if key in traces:
            raise RuntimeError(f"Duplicate reasoning trace for {key}.")
        traces[key] = row
    print(f"Reasoning traces: {len(traces):,} cases")
else:
    print("No reasoning traces from NB 15; evidence grounding (E10c) cannot be computed.")
trace_missing = sorted(set(reference_rows) - set(traces))

# Qualitative evidence channels, for the grounding check.
evidence = defaultdict(dict)
for label, path in [("A6", STAGE_B_DIR / "nb08_biomedclip_entity" / "entity_findings.jsonl"),
                    ("A4", STAGE_B_DIR / "nb11_nvreason" / "nvreason_findings.jsonl")]:
    if path.is_file():
        for row in cm.read_jsonl(path):
            key = str(row.get("image_key"))
            if label in evidence[key]:
                raise RuntimeError(f"Duplicate {label} evidence row for {key}.")
            evidence[key][label] = row

cases = []
for key, row in reference_rows.items():
    record = truth.loc[key]
    gt_total = int(record["mrale_total_annotated"])
    predicted = row.get("mrale_total")
    valid = strict_bool(row.get("valid"), False) and not cm.is_missing(predicted)
    error = (abs(float(predicted) - gt_total) if valid else cm.INVALID_TOTAL_PENALTY)
    panel = agent_totals.get(key, {})
    view = views.loc[key].to_dict() if key in views.index else {}
    trace = traces.get(key) or {}
    cases.append({
        "image_key": key, "filename": record["filename"], "fold": int(record["fold"]),
        "patient": str(record["group_id"]),
        "gt_mrale_total": gt_total, "gt_band": cm.severity_band(gt_total),
        "gt_covid": covid_label(record["covid_positive"]),
        "predicted_mrale_total": (None if not valid else int(predicted)),
        "predicted_mrale_right": row.get("mrale_right"),
        "predicted_mrale_left": row.get("mrale_left"),
        "predicted_extent_right": row.get("extent_right"),
        "predicted_density_right": row.get("density_right"),
        "predicted_extent_left": row.get("extent_left"),
        "predicted_density_left": row.get("density_left"),
        "gt_extent_right": int(record["extent_right_numerical"]),
        "gt_density_right": int(record["density_right_numerical"]),
        "gt_extent_left": int(record["extent_left_numerical"]),
        "gt_density_left": int(record["density_left_numerical"]),
        "predicted_band": (None if not valid else cm.severity_band(int(predicted))),
        "predicted_covid": covid_label(row.get("covid_pred")),
        "covid_score": row.get("covid_score"),
        "absolute_error": error, "valid_output": valid,
        "parse_error": row.get("parse_error"),
        "correct": bool(valid and error <= SAMPLING_RULE["correct_within"]),
        "n_agents": len(panel), "agent_totals": json.dumps(panel, sort_keys=True),
        "agent_sd": (float(trace.get("agent_spread"))
                     if not cm.is_missing(trace.get("agent_spread"))
                     else (float(np.std(list(panel.values()))) if len(panel) > 1 else 0.0)),
        "agent_conflict": strict_bool(trace.get("conflict_flagged"), False),
        "localization_fallback": strict_bool(view.get("any_fallback"), False),
        "laterality_plausible": strict_bool(view.get("laterality_plausible"), True),
        "v0_image": view.get("v0_image"), "v1_thorax_image": view.get("v1_thorax_image"),
        "v2_left_image": view.get("v2_left_image"),
        "v2_right_image": view.get("v2_right_image"),
        "left_box": view.get("left_box"), "right_box": view.get("right_box"),
        "rationale": trace.get("rationale"),
        "agents_cited": json.dumps(trace.get("agents_cited") or []),
        "agents_shown": json.dumps(agent_names(trace.get("agents_shown"))),
    })

cases = sorted(cases, key=lambda item: str(item["image_key"]))
case_table = pd.DataFrame(cases)
CASE_TABLE_FINGERPRINT = hashlib.sha256(json.dumps(
    cases, sort_keys=True, default=str, separators=(",", ":")).encode("utf-8")
).hexdigest()[:16]
print(f"\nCases: {len(case_table):,} (fingerprint {CASE_TABLE_FINGERPRINT})")
print(f"  correct (within {SAMPLING_RULE['correct_within']}): "
      f"{int(case_table['correct'].sum()):,}")
print(f"  localization fallbacks: {int(case_table['localization_fallback'].sum()):,}")
print(f"  agent conflicts (NB 15 validation-locked flags): "
      f"{int(case_table['agent_conflict'].sum()):,}")
print(f"  band distribution: {dict(Counter(case_table['gt_band']))}")

## 3. Execute the sampling rule

Verbatim, seeded, and stamped. The selection hash covers the rule, the locked case-table
fingerprint, and the resulting case list, so a reader can confirm the panels in the paper are
the ones the rule produced from the exact predictions rather than the ones that looked best.

A stratum with fewer cases than the quota contributes what it has, and the shortfall is recorded
— silently sampling fewer would make the figure look complete when it is not.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 20 before code cell 8")

rng = random.Random(SAMPLING_RULE["seed"])
selected, shortfalls = OrderedDict(), []


def take(frame, n, reason):
    keys = sorted(frame["image_key"].tolist())
    if len(keys) > n:
        rng.shuffle(keys)
        keys = sorted(keys[:n])
    elif len(keys) < n:
        shortfalls.append({"reason": reason, "requested": n, "available": len(keys)})
    for key in keys:
        selected.setdefault(key, []).append(reason)
    return keys


for band in ["none", "mild", "moderate", "severe"]:
    for outcome, mask in [("correct", True), ("incorrect", False)]:
        subset = case_table[(case_table["gt_band"] == band)
                            & (case_table["correct"] == mask)]
        take(subset, SAMPLING_RULE["per_band_per_outcome"], f"band_{band}_{outcome}")

largest = case_table.sort_values(
    ["absolute_error", "image_key"], ascending=[False, True]).head(
    SAMPLING_RULE["largest_errors"])
for key in largest["image_key"]:
    selected.setdefault(key, []).append("largest_error")

for key in case_table[case_table["localization_fallback"]]["image_key"]:
    selected.setdefault(key, []).append("localization_fallback")
for key in case_table[case_table["agent_conflict"]]["image_key"]:
    selected.setdefault(key, []).append("agent_conflict")

selection = case_table[case_table["image_key"].isin(selected)].copy()
selection["selection_reasons"] = selection["image_key"].map(
    lambda k: ";".join(sorted(selected[k])))
selection["_band_order"] = pd.Categorical(
    selection["gt_band"], categories=["none", "mild", "moderate", "severe"],
    ordered=True)
selection = selection.sort_values(
    ["_band_order", "absolute_error", "image_key"],
    ascending=[True, False, True]).drop(columns="_band_order")
SELECTION_HASH = hashlib.sha256(
    (RULE_HASH + "|" + CASE_TABLE_FINGERPRINT + "|"
     + "|".join(sorted(selection["image_key"]))).encode("utf-8")).hexdigest()[:16]
selection["sampling_rule_hash"] = RULE_HASH
selection["selection_hash"] = SELECTION_HASH
selection["case_table_fingerprint"] = CASE_TABLE_FINGERPRINT
selection["reference_arm"] = REFERENCE_ARM
selection.to_csv(NB20_DIR / "case_selection.csv", index=False)

print(f"Selected {len(selection):,} cases (selection hash {SELECTION_HASH})")
print(dict(Counter(reason for reasons in selected.values() for reason in reasons)))
if shortfalls:
    print()
    print("Strata with fewer cases than the quota:")
    for entry in shortfalls:
        print(f"  {entry['reason']}: requested {entry['requested']}, "
              f"available {entry['available']}")
    print("  Recorded rather than topped up from elsewhere: a panel grid that looks full when")
    print("  a stratum is empty misrepresents the cohort.")

## 4. Case panels (E10a)

Each panel: the radiograph, the localized lung boxes, the two per-lung views, the per-lung
extent/density and total, the PCR decision, and the reasoner's structured justification.

That layout is what referee 1.4 asked for — "correctly detected low, moderate, severe, and
failure cases" — with enough context that a reader can judge whether the justification matches
what the image shows.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 20 before code cell 10")

panel_paths, panel_errors = [], []
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    from PIL import Image

    Image.MAX_IMAGE_PIXELS = None
    plt.rcParams.update({"font.size": 7.5, "figure.dpi": 130, "savefig.bbox": "tight"})

    def panel_path_for(row):
        stem = re.sub(r"[^A-Za-z0-9._-]+", "_", Path(str(row["filename"])).stem)
        key_hash = hashlib.sha256(str(row["image_key"]).encode("utf-8")).hexdigest()[:10]
        return PANEL_DIR / f"panel_{key_hash}_{stem}.png"

    expected_panel_paths = {str(row["image_key"]): panel_path_for(row)
                            for row in selection.to_dict("records")}
    stale_panels = sorted(set(PANEL_DIR.glob("panel_*.png"))
                          - set(expected_panel_paths.values()))
    for stale_path in stale_panels:
        stale_path.unlink()
    if stale_panels:
        print(f"Removed {len(stale_panels)} stale generated panel(s) from an older selection.")

    def load(path):
        if not path or cm.is_missing(path) or not Path(str(path)).is_file():
            return None
        with Image.open(str(path)) as handle:
            return np.asarray(handle.convert("L"))

    for row in selection.to_dict("records"):
        base = load(row.get("v0_image"))
        if base is None:
            panel_errors.append((row["image_key"], "no readable V0 image"))
            continue
        figure, axes = plt.subplots(1, 4, figsize=(11.0, 3.4),
                                    gridspec_kw={"width_ratios": [1.25, 1.25, 0.75, 1.6]})
        axes[0].imshow(base, cmap="gray"); axes[0].set_title("Radiograph")
        axes[1].imshow(base, cmap="gray"); axes[1].set_title("A1 lung boxes")
        for side, colour in [("left_box", "tab:cyan"), ("right_box", "tab:orange")]:
            try:
                box = json.loads(row.get(side) or "null")
            except (TypeError, ValueError):
                box = None
            if box and len(box) == 4:
                axes[1].add_patch(patches.Rectangle(
                    (box[0], box[1]), box[2] - box[0], box[3] - box[1],
                    fill=False, edgecolor=colour, linewidth=1.3))
        stacked = [load(row.get("v2_right_image")), load(row.get("v2_left_image"))]
        stacked = [s for s in stacked if s is not None]
        if stacked:
            height = min(s.shape[0] for s in stacked)
            axes[2].imshow(np.hstack([s[:height] for s in stacked]), cmap="gray")
        axes[2].set_title("V2 right | left")
        for axis in axes[:3]:
            axis.set_xticks([]); axis.set_yticks([])

        agents = json.loads(row["agent_totals"] or "{}")
        predicted = row["predicted_mrale_total"]
        lines = [
            f"{row['filename']}   fold {row['fold']}",
            f"selected: {row['selection_reasons']}",
            "",
            f"mRALE truth      {row['gt_mrale_total']}  ({row['gt_band']})",
            f"mRALE predicted  {'INVALID' if predicted is None else predicted}"
            + ("" if predicted is None else f"  ({row['predicted_band']})"),
            f"absolute error   {row['absolute_error']:.0f}"
            + ("  [invalid-output penalty]" if not row["valid_output"] else ""),
            (f"right lung      extent {row['predicted_extent_right']} / density "
             f"{row['predicted_density_right']} / score {row['predicted_mrale_right']}  "
             f"[truth {row['gt_extent_right']} x {row['gt_density_right']}]"),
            (f"left lung       extent {row['predicted_extent_left']} / density "
             f"{row['predicted_density_left']} / score {row['predicted_mrale_left']}  "
             f"[truth {row['gt_extent_left']} x {row['gt_density_left']}]"),
            "",
            f"PCR label        {row['gt_covid']}",
            f"PCR predicted    {row['predicted_covid']}"
            + (f"   p={float(row['covid_score']):.2f}"
               if not cm.is_missing(row.get("covid_score")) else ""),
            "",
            "agent panel: " + (", ".join(f"{a} {v:.0f}" for a, v in sorted(agents.items()))
                               or "none"),
            f"agent SD {row['agent_sd']:.2f}"
            + ("   CONFLICT" if row["agent_conflict"] else ""),
            "localization fallback: " + ("YES" if row["localization_fallback"] else "no"),
        ]
        rationale = row.get("rationale")
        if rationale and not cm.is_missing(rationale):
            lines += ["", "reasoner justification:"]
            words, current = str(rationale).split(), ""
            for word in words:
                if len(current) + len(word) > 46:
                    lines.append("  " + current); current = word
                else:
                    current = f"{current} {word}".strip()
            if current:
                lines.append("  " + current)
        axes[3].axis("off")
        axes[3].text(0, 1, "\n".join(lines), va="top", ha="left", family="monospace",
                     fontsize=7)
        path = expected_panel_paths[str(row["image_key"])]
        figure.savefig(path); plt.close(figure)
        panel_paths.append(path)

    print(f"Rendered {len(panel_paths)} case panels to {PANEL_DIR}")
    if panel_errors:
        print(f"  {len(panel_errors)} case(s) could not be rendered, e.g. {panel_errors[:2]}")
except ImportError:
    print("matplotlib or PIL unavailable; case_selection.csv is complete and panels can be "
          "rendered later without re-running the sampling rule.")

## 5. Failure taxonomy (E10b) — automatic first pass, then two humans

The categories are pre-declared. The automatic pass assigns whichever it can determine from the
data; the rest are left blank for human coding.

| code | meaning | determinable automatically? |
| --- | --- | --- |
| `schema_violation` | output could not be parsed | yes |
| `localization_error` | box implausible or fell back to a heuristic | yes |
| `laterality_swap` | right/left errors are large and opposite in sign | yes |
| `extent_overcall` / `extent_undercall` | extent components too high/low | yes, when components are emitted |
| `density_overcall` / `density_undercall` | density components too high/low | yes, when components are emitted |
| `pcr_label_vs_imaging_mismatch` | PCR-positive, clean radiograph, model agrees with the image | yes |
| `severity_misjudgement` | wrong band with no more specific cause | residual |
| `no_error` | within tolerance | yes |

**`pcr_label_vs_imaging_mismatch` is the category that matters.** A PCR-positive patient with a
radiograph showing nothing is not a model failure. Counting it as one inflates the error rate
and hides the real explanation for why detection does not work on this cohort.

The automatic pass is **coder A only**. The gate below refuses to call the taxonomy dual-coded
until a second human coding exists, because κ between an algorithm and itself is 1.0 and means
nothing.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 20 before code cell 12")

TAXONOMY = ["no_error", "schema_violation", "localization_error", "laterality_swap",
            "extent_overcall", "extent_undercall", "density_overcall", "density_undercall",
            "pcr_label_vs_imaging_mismatch", "severity_misjudgement"]

# A radiograph is "clean" when the annotators scored it 0. That is the reference standard for
# whether there was anything to see.
CLEAN_RADIOGRAPH_TOTAL = 0
LATERALITY_SWAP_MIN_ERROR = 4      # each side must be wrong by this much, in opposite directions

component_rows = {str(r["image_key"]): r for r in raw_rows if r["arm"] == REFERENCE_ARM}


def automatic_code(row):
    key = row["image_key"]
    record = component_rows.get(key, {})
    if not row["valid_output"]:
        return "schema_violation", "output could not be parsed"
    if row["localization_fallback"] or row.get("laterality_plausible") is False:
        return "localization_error", "heuristic fallback or implausible lung boxes"

    truth_row = truth.loc[key]
    gt_right = int(truth_row["extent_right_numerical"]) * int(truth_row["density_right_numerical"])
    gt_left = int(truth_row["extent_left_numerical"]) * int(truth_row["density_left_numerical"])
    predicted_right, predicted_left = record.get("mrale_right"), record.get("mrale_left")
    if not cm.is_missing(predicted_right) and not cm.is_missing(predicted_left):
        error_right = float(predicted_right) - gt_right
        error_left = float(predicted_left) - gt_left
        native_error = abs(float(predicted_right) - gt_right) + abs(float(predicted_left) - gt_left)
        swapped_error = abs(float(predicted_right) - gt_left) + abs(float(predicted_left) - gt_right)
        if (abs(error_right) >= LATERALITY_SWAP_MIN_ERROR
                and abs(error_left) >= LATERALITY_SWAP_MIN_ERROR
                and error_right * error_left < 0
                and swapped_error + 1e-6 < native_error):
            return "laterality_swap", (f"right {error_right:+.0f}, left {error_left:+.0f} — "
                                       f"large and opposite; swapped error {swapped_error:.1f} "
                                       f"< native error {native_error:.1f}")

    # PCR false negative with mRALE 0, where the severity estimate matched the opacity label.
    if (row["gt_covid"] == "Yes" and row["predicted_covid"] == "No"
            and row["gt_mrale_total"] == CLEAN_RADIOGRAPH_TOTAL
            and row["predicted_mrale_total"] is not None
            and int(row["predicted_mrale_total"]) <= 2):
        return "pcr_label_vs_imaging_mismatch", ("PCR-positive with an annotated-clean "
                                                 "radiograph; the model matched the image")

    if row["correct"]:
        return "no_error", f"within {SAMPLING_RULE['correct_within']} mRALE points"

    for component, high_code, low_code in [("extent", "extent_overcall", "extent_undercall"),
                                           ("density", "density_overcall",
                                            "density_undercall")]:
        deltas = []
        for side in ["right", "left"]:
            predicted = record.get(f"{component}_{side}")
            reference = truth_row.get(f"{component}_{side}_numerical")
            if not cm.is_missing(predicted) and not cm.is_missing(reference):
                deltas.append(float(predicted) - float(reference))
        if deltas and abs(sum(deltas)) >= 2:
            return (high_code if sum(deltas) > 0 else low_code,
                    f"{component} components off by {sum(deltas):+.0f} in total")
    return "severity_misjudgement", "wrong band with no more specific cause identified"


sheet_path = NB20_DIR / "coding_sheet.csv"
existing_coding = pd.read_csv(sheet_path) if sheet_path.is_file() else pd.DataFrame()
coding_rows = []
for row in selection.to_dict("records"):
    code_value, reason = automatic_code(row)
    coding_rows.append({
        "image_key": row["image_key"], "filename": row["filename"],
        "selection_reasons": row["selection_reasons"], "gt_band": row["gt_band"],
        "gt_mrale_total": row["gt_mrale_total"],
        "predicted_mrale_total": row["predicted_mrale_total"],
        "absolute_error": row["absolute_error"], "gt_covid": row["gt_covid"],
        "predicted_covid": row["predicted_covid"],
        "selection_hash": SELECTION_HASH,
        "case_table_fingerprint": CASE_TABLE_FINGERPRINT,
        "coder_auto": code_value, "coder_auto_reason": reason,
        "coder_A_human": "", "coder_B_human": "", "adjudicated": ""})

coding_sheet = pd.DataFrame(coding_rows)
if len(existing_coding):
    if "image_key" not in existing_coding:
        raise RuntimeError("Existing coding_sheet.csv has no image_key column.")
    if existing_coding["image_key"].astype(str).duplicated().any():
        raise RuntimeError("Existing coding_sheet.csv has duplicate image_key rows.")
    filled_columns = [column for column in ["coder_A_human", "coder_B_human", "adjudicated"]
                      if column in existing_coding]
    has_manual = any(existing_coding[column].fillna("").astype(str).str.strip().ne("").any()
                     for column in filled_columns)
    old_keys = set(existing_coding["image_key"].astype(str))
    new_keys = set(coding_sheet["image_key"].astype(str))
    old_fingerprints = (set(existing_coding["case_table_fingerprint"].dropna().astype(str))
                        if "case_table_fingerprint" in existing_coding else set())
    provenance_changed = (old_keys != new_keys
                          or old_fingerprints != {CASE_TABLE_FINGERPRINT})
    if has_manual and provenance_changed:
        raise RuntimeError(
            "The case selection or its prediction/trace fingerprint changed after human "
            "coding began. Preserve the existing sheet and resolve the input change "
            "explicitly; NB 20 will not attach old judgements to new panels.")
    if not provenance_changed:
        preserved = existing_coding.set_index(existing_coding["image_key"].astype(str))
        for column in ["coder_A_human", "coder_B_human", "adjudicated"]:
            if column in preserved:
                coding_sheet[column] = coding_sheet["image_key"].astype(str).map(preserved[column])
coding_sheet.to_csv(sheet_path, index=False)
taxonomy_counts = Counter(coding_sheet["coder_auto"])
failure_taxonomy = pd.DataFrame(
    [{"code": code, "n": taxonomy_counts.get(code, 0),
      "fraction_of_selection": round(taxonomy_counts.get(code, 0) / max(len(coding_sheet), 1), 4),
      "coder": "automatic first pass"} for code in TAXONOMY])
failure_taxonomy.to_csv(NB20_DIR / "failure_taxonomy.csv", index=False)

print(failure_taxonomy[failure_taxonomy["n"] > 0].to_string(index=False))
print()
print(f"coding_sheet.csv has {len(coding_sheet)} rows with empty coder_A_human / coder_B_human")
print("columns. Two authors fill those independently; re-run this notebook to compute kappa.")

mismatch = taxonomy_counts.get("pcr_label_vs_imaging_mismatch", 0)
if mismatch:
    print()
    print(f"{mismatch} selected PCR false-negative case(s) have mRALE 0 and a model "
          "estimate within 2 points of that opacity annotation.")
    print("  These are not mRALE-opacity errors when the model also scored zero; the COVID")
    print("  decision can still be wrong and must be evaluated against PCR separately.")

## 6. Cohort-wide: how much of the detection problem is the label?

The taxonomy above covers the sampled panel. This applies the same PCR-versus-imaging test to
the **whole** cohort, because the fraction of PCR-positive cases with a clean radiograph is the
single most useful number for interpreting the detection result.

A large share of PCR-positive images with mRALE 0 establishes PCR/opacity discordance. It does
not show that every other radiographic feature is absent or that no image architecture can
separate the groups, so the interpretation remains deliberately bounded.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 20 before code cell 14")

cohort = case_table.copy()
positives = cohort[cohort["gt_covid"] == "Yes"]
negatives = cohort[cohort["gt_covid"] == "No"]

label_rows = []
if len(positives):
    clean_positives = positives[positives["gt_mrale_total"] == CLEAN_RADIOGRAPH_TOTAL]
    label_rows.append({
        "group": "PCR-positive", "n": len(positives),
        "n_clean_radiograph": len(clean_positives),
        "fraction_clean": round(len(clean_positives) / len(positives), 4),
        "median_mrale": float(positives["gt_mrale_total"].median())})
if len(negatives):
    abnormal_negatives = negatives[negatives["gt_mrale_total"] > 0]
    label_rows.append({
        "group": "PCR-negative", "n": len(negatives),
        "n_abnormal_radiograph": len(abnormal_negatives),
        "fraction_abnormal": round(len(abnormal_negatives) / len(negatives), 4),
        "median_mrale": float(negatives["gt_mrale_total"].median())})

label_analysis = pd.DataFrame(label_rows)
label_analysis.to_csv(NB20_DIR / "label_vs_imaging.csv", index=False)
print(label_analysis.to_string(index=False))

if len(positives) and len(negatives):
    clean_positive_fraction = float((positives["gt_mrale_total"]
                                     == CLEAN_RADIOGRAPH_TOTAL).mean())
    abnormal_negative_fraction = float((negatives["gt_mrale_total"] > 0).mean())
    discordant_count = clean_positive_fraction * len(positives) + \
        abnormal_negative_fraction * len(negatives)
    print()
    print(f"{clean_positive_fraction:.1%} of PCR-positive cases have an annotated-clean "
          "radiograph.")
    print(f"{abnormal_negative_fraction:.1%} of PCR-negative cases have visible opacity.")
    print(f"Together, {discordant_count / len(cohort):.1%} of the cohort has an opacity/PCR "
          "discordance under the available mRALE annotation.")
    print()
    print("  The mRALE opacity annotation alone cannot establish whether other image findings")
    print("  separate these PCR groups. Report this as label/opacity discordance, not as an")
    print("  irreducible ceiling for every possible image-based model.")

## 7. Evidence grounding (E10c)

Three rates over the reasoner's stated justifications:

- **grounding rate** — the justification references a finding an agent actually reported, or a
  lung box that exists;
- **unsupported-claim rate** — it names a finding no agent reported;
- **contradiction rate** — it asserts something the MIDRC annotation contradicts (claiming
  bilateral opacity on a radiograph annotated 0, for example).

This is a keyword-level check and it is described as one. It bounds the problem rather than
settling it: a high unsupported rate is strong evidence of a problem, a low one is weak evidence
of its absence, and the reader study in section 8 is what would settle it.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 20 before code cell 16")

FINDING_VOCABULARY = [
    "consolidation", "ground glass", "ground-glass", "opacity", "opacities", "effusion",
    "atelectasis", "edema", "oedema", "infiltrate", "infiltration", "nodule", "reticular",
    "interstitial", "pneumothorax", "cardiomegaly", "hazy", "haziness", "airspace",
]
OPACITY_TERMS = {
    "consolidation", "ground glass", "ground-glass", "opacity", "opacities",
    "edema", "oedema", "infiltrate", "infiltration", "hazy", "haziness",
    "airspace",
}
LATERAL_TERMS = {"bilateral": "both", "left": "left", "right": "right", "diffuse": "both"}
FINDING_ALIASES = {
    "ground-glass": "ground glass", "opacities": "opacity",
    "oedema": "edema", "infiltration": "infiltrate",
    "haziness": "hazy",
}
NEGATION_PATTERN = re.compile(
    r"(?:\bno\b|\bwithout\b|\babsent\b|\bnegative for\b|\bfree of\b)"
    r"(?:\W+\w+){0,3}\W*$", re.IGNORECASE)


def canonical_finding(value):
    text = str(value).strip().lower().replace("_", " ")
    return FINDING_ALIASES.get(text, text)


def rationale_findings(text):
    """Separate asserted findings from locally negated findings."""
    asserted, negated = set(), set()
    for term in FINDING_VOCABULARY:
        for match in re.finditer(rf"(?<!\w){re.escape(term)}(?!\w)", text, re.IGNORECASE):
            prefix = text[max(0, match.start() - 55):match.start()]
            target = negated if NEGATION_PATTERN.search(prefix) else asserted
            target.add(canonical_finding(term))
    return sorted(asserted), sorted(negated)

grounding_rows = []
for row in selection.to_dict("records"):
    key = row["image_key"]
    rationale = row.get("rationale")
    has_rationale = not cm.is_missing(rationale) and bool(str(rationale).strip())
    # A missing justification is an E10c failure mode, not a reason to remove the case
    # from the denominator. Retain it with zero mentioned/supported findings.
    text = str(rationale).lower() if has_rationale else ""
    mentioned, negated = rationale_findings(text)
    reported = set()
    for channel in (evidence.get(key) or {}).values():
        for finding in (channel.get("findings") or []):
            value = finding.get("finding") if isinstance(finding, dict) else finding
            if value is not None:
                normalized = canonical_finding(value)
                if normalized:
                    reported.add(normalized)
    reported_text = " ".join(reported)

    supported_by_agent = [t for t in mentioned
                          if any(t in f or f in t for f in reported) or t in reported_text]
    # The region-level mRALE annotation directly supports or contradicts an asserted opacity
    # claim. It does not adjudicate unrelated findings such as effusion or cardiomegaly.
    supported_by_annotation = [
        t for t in mentioned if t in {canonical_finding(v) for v in OPACITY_TERMS}
        and float(row["gt_mrale_total"]) > CLEAN_RADIOGRAPH_TOTAL]
    supported = sorted(set(supported_by_agent) | set(supported_by_annotation))
    unsupported = [t for t in mentioned if t not in supported]
    cited = json.loads(row.get("agents_cited") or "[]")
    panel = json.loads(row.get("agent_totals") or "{}")
    trace_panel = json.loads(row.get("agents_shown") or "[]")
    present_agents = set(panel) | set(trace_panel) | set((evidence.get(key) or {}).keys())
    cited_present = [a for a in cited if a in present_agents]

    # mRALE can contradict opacity claims only; it does not annotate other findings.
    canonical_opacity = {canonical_finding(v) for v in OPACITY_TERMS}
    opacity_claims = sorted(set(mentioned) & canonical_opacity)
    negated_opacity = sorted(set(negated) & canonical_opacity)
    contradiction = bool(
        (opacity_claims and row["gt_mrale_total"] == CLEAN_RADIOGRAPH_TOTAL)
        or (negated_opacity and row["gt_mrale_total"] > CLEAN_RADIOGRAPH_TOTAL))
    grounding_rows.append({
        "image_key": key, "gt_mrale_total": row["gt_mrale_total"],
        "has_rationale": has_rationale,
        "n_findings_mentioned": len(mentioned), "findings_mentioned": ";".join(mentioned),
        "findings_negated": ";".join(negated),
        "n_supported": len(supported), "n_unsupported": len(unsupported),
        "supported_by_agent": ";".join(sorted(supported_by_agent)),
        "supported_by_annotation": ";".join(sorted(supported_by_annotation)),
        "unsupported_findings": ";".join(unsupported),
        "opacity_claims": ";".join(opacity_claims),
        "negated_opacity_claims": ";".join(negated_opacity),
        "n_agents_cited": len(cited), "n_cited_present_in_panel": len(cited_present),
        "cites_only_present_agents": (None if not cited else
                                      len(cited_present) == len(cited)),
        "contradicts_annotation": contradiction,
        "has_evidence_channel": key in evidence})

GROUNDING_COLUMNS = [
    "image_key", "gt_mrale_total", "has_rationale", "n_findings_mentioned",
    "findings_mentioned", "findings_negated", "n_supported", "n_unsupported",
    "supported_by_agent", "supported_by_annotation", "unsupported_findings",
    "opacity_claims", "negated_opacity_claims", "n_agents_cited",
    "n_cited_present_in_panel", "cites_only_present_agents",
    "contradicts_annotation", "has_evidence_channel"]
grounding = pd.DataFrame(grounding_rows).reindex(columns=GROUNDING_COLUMNS)
# Always replace the generated artifact so a no-trace rerun cannot leave an older,
# apparently successful grounding table in place.
grounding.to_csv(NB20_DIR / "grounding_metrics.csv", index=False)
if len(grounding):
    with_rationale = grounding[grounding["has_rationale"].astype(bool)]
    with_findings = with_rationale[with_rationale["n_findings_mentioned"] > 0]
    summary = {
        "n_selected_cases": int(len(grounding)),
        "n_rationales": int(len(with_rationale)),
        "n_missing_rationales": int(len(grounding) - len(with_rationale)),
        "rationale_coverage": round(float(len(with_rationale) / len(grounding)), 4),
        "n_with_a_named_finding": int(len(with_findings)),
        "grounding_rate": (round(float((with_findings["n_supported"] > 0).mean()), 4)
                           if len(with_findings) else None),
        "unsupported_claim_rate": (round(float((with_findings["n_unsupported"] > 0).mean()), 4)
                                   if len(with_findings) else None),
        "contradiction_rate": (
            round(float(with_rationale["contradicts_annotation"].mean()), 4)
            if len(with_rationale) else None),
        "citation_coverage": (
            round(float((with_rationale["n_agents_cited"] > 0).mean()), 4)
            if len(with_rationale) else None),
        "citation_validity_rate": (
            round(float(with_rationale.loc[with_rationale["n_agents_cited"] > 0,
                                      "cites_only_present_agents"].astype(bool).mean()), 4)
            if (with_rationale["n_agents_cited"] > 0).any() else None),
        "coverage_of_evidence_channels": round(float(grounding["has_evidence_channel"].mean()), 4),
        "available": True,
        "method": ("negation-aware keyword matching against agent finding channels, "
                   "with opacity assertions checked against regional mRALE annotations; "
                   "bounds the problem, does not settle it"),
    }
    sd.write_json_atomic(NB20_DIR / "grounding_summary.json", summary)
    print(json.dumps(summary, indent=2))
    if summary["n_with_a_named_finding"] < 0.2 * max(len(with_rationale), 1):
        print()
        print("Most rationales name no radiographic finding at all. A justification that says")
        print("nothing specific cannot be checked for grounding, and cannot support an")
        print("auditability claim either — which is a finding about the system, not the check.")
else:
    sd.write_json_atomic(NB20_DIR / "grounding_summary.json", {
        "available": False, "n_rationales": 0,
        "reason": "No non-empty reasoner rationales were available for selected cases."})
    print("No rationales were available; E10c cannot be computed. NB 15's reasoning traces are "
          "the input.")

## 8. Dual coding and Cohen's κ (E10b)

If two human codings exist in `coding_sheet.csv`, κ is computed and reported. If they do not,
the notebook says so and the gate blocks the "dual-coded" description.

κ between an automatic pass and itself is 1.0 and means nothing. Reporting that as inter-rater
agreement would be a misrepresentation of the method, so the code will not produce it.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 20 before code cell 18")

kappa_result = {"available": False,
                "reason": "coder_A_human / coder_B_human are empty in coding_sheet.csv"}
sheet_path = NB20_DIR / "coding_sheet.csv"
existing = pd.read_csv(sheet_path) if sheet_path.is_file() else pd.DataFrame()

if len(existing) and {"coder_A_human", "coder_B_human"} <= set(existing.columns):
    paired = existing.dropna(subset=["coder_A_human", "coder_B_human"]).copy()
    paired["coder_A_human"] = paired["coder_A_human"].astype(str).str.strip()
    paired["coder_B_human"] = paired["coder_B_human"].astype(str).str.strip()
    if "adjudicated" in paired:
        paired["adjudicated"] = paired["adjudicated"].fillna("").astype(str).str.strip()
    paired = paired[(paired["coder_A_human"] != "")
                    & (paired["coder_B_human"] != "")]
    if len(paired) == len(existing) and len(paired) >= 10:
        invalid_codes = ((set(paired["coder_A_human"].astype(str))
                          | set(paired["coder_B_human"].astype(str))) - set(TAXONOMY))
        if invalid_codes:
            raise ValueError(f"Human coding contains categories outside TAXONOMY: {sorted(invalid_codes)}")
        result = sd.cohens_kappa(paired["coder_A_human"].astype(str).tolist(),
                                 paired["coder_B_human"].astype(str).tolist(),
                                 categories=TAXONOMY)
        disagreements = paired[paired["coder_A_human"].astype(str)
                               != paired["coder_B_human"].astype(str)]
        kappa_result = {
            "available": True, "n_dual_coded": int(len(paired)),
            "kappa": round(result["kappa"], 4),
            "observed_agreement": round(result["observed_agreement"], 4),
            "interpretation": result.get("interpretation"),
            "n_disagreements": int(len(disagreements)),
            "n_adjudicated": int((existing["adjudicated"].fillna("").astype(str).str.strip() != "").sum())
            if "adjudicated" in existing.columns else 0,
            "categories": TAXONOMY, "taxonomy_final_available": False}
        final_codes = []
        unresolved = []
        for _, coded in paired.iterrows():
            code_a = str(coded["coder_A_human"]).strip()
            code_b = str(coded["coder_B_human"]).strip()
            adjudicated = str(coded.get("adjudicated", "") or "").strip()
            if code_a == code_b:
                final_codes.append(code_a)
            elif adjudicated in TAXONOMY:
                final_codes.append(adjudicated)
            else:
                unresolved.append(str(coded["image_key"]))
        if not unresolved:
            taxonomy_counts = Counter(final_codes)
            failure_taxonomy = pd.DataFrame([
                {"code": code, "n": taxonomy_counts.get(code, 0),
                 "fraction_of_selection": round(
                     taxonomy_counts.get(code, 0) / max(len(final_codes), 1), 4),
                 "coder": "two independent humans; disagreements adjudicated"}
                for code in TAXONOMY])
            failure_taxonomy.to_csv(NB20_DIR / "failure_taxonomy.csv", index=False)
            kappa_result["taxonomy_final_available"] = True
        else:
            kappa_result["unresolved_disagreements"] = unresolved
        print(json.dumps(kappa_result, indent=2))
        if result["kappa"] < 0.6:
            print()
            print("Kappa below 0.6 is moderate at best. Either the category definitions are")
            print("ambiguous or the coders applied them differently; resolve that before the")
            print("taxonomy carries any weight in the manuscript.")
    else:
        kappa_result["reason"] = (f"{len(paired)}/{len(existing)} rows are fully dual-coded; "
                                  "all selected rows (and at least 10) are required")

if not kappa_result["available"]:
    print("Dual coding is NOT yet available.")
    print(f"  {kappa_result['reason']}")
    print()
    print("  To complete E10b: two authors independently fill coder_A_human and coder_B_human")
    print(f"  in {sheet_path}, using the categories in failure_taxonomy.csv, then re-run this")
    print("  notebook. The automatic pass is a starting point for coder A, not a substitute")
    print("  for either coder — kappa against an algorithm's own output is 1.0 by construction.")

sd.write_json_atomic(NB20_DIR / "dual_coding_kappa.json", kappa_result)

## 9. Reader-study protocol (E10d) — specified, not executed

Referee 1 asked for this "even as future work", so it is written properly rather than gestured
at. The document below is generated with the sample size actually computed, not asserted.

It is explicitly **declared future work**: nothing in this revision claims a reader study was
run.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 20 before code cell 20")

# Sample size for detecting a 0.5-point difference on a 5-point Likert scale, paired design.
EFFECT = 0.5
ASSUMED_SD = 1.0            # conservative for a 5-point scale
POWER, ALPHA_TWO_SIDED = 0.80, 0.05
z_alpha = 1.959963984540054
z_beta = 0.8416212335729143
n_per_reader = math.ceil(((z_alpha + z_beta) ** 2 * ASSUMED_SD ** 2) / (EFFECT ** 2))
N_READERS, N_CASES = 3, 60
effective_cases = N_CASES

protocol_text = f"""# Appendix B — reader-study protocol (declared future work, not executed)

Referee 1 asked for a reader study "even as future work". This specifies one; it was **not**
run for this revision, and no claim in the manuscript rests on it.

## Design
Paired, blinded, within-reader comparison of two systems' justifications for the same
radiographs.

- **Readers**: {N_READERS} board-certified radiologists, independent of the development team.
- **Cases**: {N_CASES} radiographs drawn by the pre-registered E10 sampling rule
  (hash `{RULE_HASH}`, selection hash `{SELECTION_HASH}`), stratified across the four severity
  bands and across correct/incorrect model outcomes.
- **Blinding**: system identity hidden; presentation order randomised per reader; the two
  systems' outputs for a case appear side by side in randomised left/right position.

## Instrument
Per case and per system, a 5-point Likert rating on three axes:

1. **Usefulness** — would this justification help you reach or check a decision?
2. **Correctness** — is the justification consistent with what the radiograph shows?
3. **Trust** — would you be willing to act on this system's output given this justification?

Free-text disagreement notes are collected for every rating of 2 or below.

## Sample size
To detect a {EFFECT}-point mean difference on a 5-point scale at
{int(POWER * 100)}% power and two-sided alpha = {ALPHA_TWO_SIDED}, assuming a paired standard
deviation of {ASSUMED_SD:.1f}, requires **{n_per_reader} paired observations**. The design
supplies **{N_CASES} unique paired cases per reader**. The naive count of
{N_READERS} x {N_CASES} ratings is not treated as {N_READERS * N_CASES} independent
observations because readers and cases are crossed. Since {N_CASES} exceeds the
single-reader paired requirement of {n_per_reader}, the design is provisionally adequate,
but the final sample size must be confirmed by simulation under plausible reader and case
variance components before the study is executed.

If the observed paired SD is larger than {ASSUMED_SD:.1f}, the study is powered for a
difference of {EFFECT * math.sqrt(n_per_reader / effective_cases):.2f} points or larger
under the conservative unique-case approximation; report the
achieved sensitivity rather than the planned one.

## Analysis
- Primary: mixed-effects model with reader and case as crossed random effects, system as a
  fixed effect. Report the estimated difference with a 95% interval.
- Inter-reader agreement: intraclass correlation (two-way random, absolute agreement).
- Secondary: proportion of cases where a reader rated correctness <= 2, by severity band.
- No p-value is reported without its test name, the paired unit, the family and its size
  (protocol 8.8).

## Pre-registration
The instrument, sample size and analysis are fixed before any rating is collected. Deviations
go in the protocol-deviation log with a date and a reason.
"""
(NB20_DIR / "reader_study_protocol.md").write_text(protocol_text, encoding="utf-8")
print(f"reader_study_protocol.md written ({len(protocol_text):,} bytes)")
print(f"  {n_per_reader} paired cases required under the simple approximation; "
      f"the design supplies {effective_cases} unique cases per reader.")

## 10. Run configuration and gate

Blocking conditions:

1. **All true data producers passed their gates.** NB 04, NB 13, NB 15, and NB 17 are
   verified before their localization, registry, trace, or reference-arm artifacts are read.
2. **The panels shown are the panels the rule selects.** Re-executing the rule must reproduce
   the same selection hash. If it does not, the figure is a curated selection and cannot be
   described as pre-registered.
3. **The taxonomy is not described as dual-coded until it is.** A single automatic pass is a
   starting point, not two independent coders.
4. **Every reference case has an NB 15 trace and every selected rationale enters E10c.**
5. **Human work is attached to immutable inputs.** Existing coding is preserved only when
   both selected image keys and the complete case-table fingerprint are unchanged.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 20 before code cell 22")

failures, warnings = [], []

if not len(selection):
    failures.append("The pre-registered sampling rule selected no cases.")
if missing_views:
    failures.append(
        f"NB 04 localization/view records are missing for {len(missing_views)} reference "
        f"cases (e.g. {missing_views[:5]}).")
if trace_missing:
    failures.append(f"NB 15 reasoning traces are missing for {len(trace_missing)} reference "
                    f"cases (e.g. {trace_missing[:5]}), so E10c grounding coverage is incomplete.")
if len(grounding) != len(selection):
    failures.append(f"Grounding was computed for {len(grounding)}/{len(selection)} selected "
                    "cases; every displayed rationale must enter the E10c audit.")

# Gate 1: re-execute the rule and compare hashes.
verify_rng = random.Random(SAMPLING_RULE["seed"])
verify_selected = {}


def verify_take(frame, n, reason):
    keys = sorted(frame["image_key"].tolist())
    if len(keys) > n:
        verify_rng.shuffle(keys)
        keys = sorted(keys[:n])
    for key in keys:
        verify_selected.setdefault(key, []).append(reason)


for band in ["none", "mild", "moderate", "severe"]:
    for outcome, mask in [("correct", True), ("incorrect", False)]:
        verify_take(case_table[(case_table["gt_band"] == band)
                               & (case_table["correct"] == mask)],
                    SAMPLING_RULE["per_band_per_outcome"], f"band_{band}_{outcome}")
for key in case_table.sort_values(
        ["absolute_error", "image_key"], ascending=[False, True]).head(
        SAMPLING_RULE["largest_errors"])["image_key"]:
    verify_selected.setdefault(key, []).append("largest_error")
for key in case_table[case_table["localization_fallback"]]["image_key"]:
    verify_selected.setdefault(key, []).append("localization_fallback")
for key in case_table[case_table["agent_conflict"]]["image_key"]:
    verify_selected.setdefault(key, []).append("agent_conflict")

verify_hash = hashlib.sha256(
    (RULE_HASH + "|" + CASE_TABLE_FINGERPRINT + "|"
     + "|".join(sorted(verify_selected))).encode("utf-8")).hexdigest()[:16]
if verify_hash != SELECTION_HASH:
    failures.append(
        f"Re-executing the sampling rule produced a different selection ({verify_hash} vs "
        f"{SELECTION_HASH}). The panels cannot be described as pre-registered, because the "
        "rule does not reproduce them.")
else:
    print(f"SAMPLING RULE VERIFIED: re-execution reproduces selection {SELECTION_HASH} "
          f"({len(selection)} cases).")

selection_fingerprints = set(selection["case_table_fingerprint"].astype(str))
if selection_fingerprints != {CASE_TABLE_FINGERPRINT}:
    failures.append(
        f"case_selection.csv carries the wrong case-table fingerprint: "
        f"{sorted(selection_fingerprints)} vs {CASE_TABLE_FINGERPRINT}.")
selection_hashes = set(selection["selection_hash"].astype(str))
if selection_hashes != {SELECTION_HASH}:
    failures.append(
        f"case_selection.csv carries inconsistent selection hashes: "
        f"{sorted(selection_hashes)} vs {SELECTION_HASH}.")
empty_agent_panels = int((selection["n_agents"] == 0).sum())
if empty_agent_panels:
    failures.append(f"{empty_agent_panels} selected cases have an empty locked agent panel.")
invalid_covid_labels = case_table["gt_covid"].isna().sum()
if invalid_covid_labels:
    failures.append(f"{invalid_covid_labels} locked cases have an unrecognised PCR label.")

# Gate 2: dual coding honesty.
if not kappa_result["available"]:
    warnings.append(
        "The failure taxonomy has ONE automatic coding pass. Protocol E10b requires two "
        "independent human coders and Cohen's kappa. Until coding_sheet.csv is filled in, the "
        "manuscript must describe this as an automatic first pass, not as a dual-coded "
        "taxonomy.")
    if "dual" in "".join(str(v) for v in failure_taxonomy.get("coder", [])).lower():
        failures.append("failure_taxonomy.csv claims dual coding while no second coding "
                        "exists.")
elif not kappa_result.get("taxonomy_final_available", False):
    warnings.append("Dual coding is complete, but disagreements remain unadjudicated; "
                    "failure_taxonomy.csv remains the automatic first pass.")
elif kappa_result.get("kappa", 0) < 0.4:
    warnings.append(f"Cohen's kappa is {kappa_result['kappa']:.3f} ({kappa_result.get('interpretation')}). "
                    "The categories are not being applied consistently; resolve the definitions "
                    "before the taxonomy carries weight.")

# ---- Warnings that shape the manuscript ---------------------------------------------------
if shortfalls:
    warnings.append(
        f"{len(shortfalls)} stratum/strata had fewer cases than the quota: "
        + ", ".join(f"{s['reason']} ({s['available']}/{s['requested']})" for s in shortfalls[:4])
        + ". The panel grid is not full, and the caption should say which cells are short.")

if not panel_paths:
    warnings.append("No case panels were rendered. R1.4 asked specifically for correctly "
                    "detected low/moderate/severe and failure cases, so Figure 6 is missing.")
elif panel_errors:
    warnings.append(f"{len(panel_errors)} selected case(s) could not be rendered "
                    f"(e.g. {panel_errors[0]}).")
if len(set(panel_paths)) != len(panel_paths):
    failures.append("Two selected cases resolved to the same panel filename.")

mismatch_count = taxonomy_counts.get("pcr_label_vs_imaging_mismatch", 0)
if len(positives):
    clean_fraction = float((positives["gt_mrale_total"] == CLEAN_RADIOGRAPH_TOTAL).mean())
    if clean_fraction >= 0.05:
        warnings.append(
            f"{clean_fraction:.1%} of PCR-positive cases have an annotated-clean radiograph "
            f"({mismatch_count} appear in the sampled panel). This supports a bounded claim "
            "about PCR/mRALE-opacity discordance; it does not establish an irreducible "
            "ceiling for all image features or architectures.")
    else:
        warnings.append(
            f"Only {clean_fraction:.1%} of PCR-positive cases have an annotated-clean "
            "radiograph, so label-versus-imaging mismatch does NOT explain the detection "
            "result on this cohort. Whatever limits detection here has to be looked for "
            "elsewhere — do not reach for this explanation by default.")

if len(grounding):
    missing_rationales = int((~grounding["has_rationale"].astype(bool)).sum())
    if missing_rationales:
        warnings.append(
            f"{missing_rationales}/{len(grounding)} selected cases have no reasoner "
            "justification. They remain in the E10c denominator as ungrounded outputs; "
            "report rationale coverage rather than silently excluding them.")
    with_findings = grounding[grounding["n_findings_mentioned"] > 0]
    if len(with_findings):
        unsupported_rate = float((with_findings["n_unsupported"] > 0).mean())
        if unsupported_rate > 0.2:
            warnings.append(
                f"{unsupported_rate:.1%} of justifications that name a finding name one no "
                "agent reported. An auditability claim needs this number in the paper.")
    cited_grounding = grounding[(grounding["has_rationale"].astype(bool))
                                  & (grounding["n_agents_cited"] > 0)]
    if len(cited_grounding) and float(cited_grounding["cites_only_present_agents"].astype(bool).mean()) < 0.95:
        warnings.append("Some justifications cite agents that were not in their panel. "
                        "Citations that do not correspond to inputs are not provenance.")
else:
    warnings.append("Evidence grounding (E10c) was not computed: NB 15 produced no reasoning "
                    "traces.")

sd.write_json_atomic(NB20_DIR / "run_config.json", sd.provenance_stamp(
    "20_interpretability_and_failure_cases.ipynb",
    {"reference_arm": REFERENCE_ARM, "upstream_gates": UPSTREAM_GATES,
     "nb15_full_roster_arm": nb15_config.get("full_roster_arm"),
     "sampling_rule": SAMPLING_RULE,
     "sampling_rule_hash": RULE_HASH, "selection_hash": SELECTION_HASH,
     "case_table_fingerprint": CASE_TABLE_FINGERPRINT,
     "n_cases_available": int(len(case_table)), "n_cases_selected": int(len(selection)),
     "conflict_source": "NB 15 fold-specific validation-locked conflict_flagged",
     "shortfalls": shortfalls, "n_panels_rendered": len(panel_paths),
     "panel_errors": panel_errors,
     "taxonomy": TAXONOMY, "taxonomy_counts": dict(taxonomy_counts),
     "dual_coding": kappa_result,
     "reader_study": "specified in reader_study_protocol.md; DECLARED FUTURE WORK, not run",
     "grounding_method": ("negation-aware finding matching against agent channels, "
                          "with opacity claims checked against mRALE annotations; "
                          "bounds the problem rather than settling it")}))


def report(title, messages):
    print(title)
    for message in messages or []:
        print("  -", message)
    if not messages:
        print("  none")


print()
report("WARNINGS", warnings)
print()
report("FAILURES", failures)
sd.write_json_atomic(NB20_DIR / "gate_nb20.json",
                     {"passed": not failures, "failures": failures,
                      "warnings": warnings, "upstream_gates": UPSTREAM_GATES,
                      "case_table_fingerprint": CASE_TABLE_FINGERPRINT,
                      "selection_hash": SELECTION_HASH})
if failures:
    detail = "\n".join(f"  [{i + 1}] {m}" for i, m in enumerate(failures))
    raise AssertionError(f"NB 20 gate failed with {len(failures)} blocking issue(s):\n{detail}")
print()
print("NB 20 gate: PASSED")